1. Load heart disease dataset in pandas dataframe
2. Remove outliers using Z score. Usual guideline is to remove anything that has Z score > 3 formula or Z score < -3
3. Convert text columns to numbers using label encoding and one hot encoding
Apply scaling
4. Build a classification model using various methods (SVM, logistic regression, random forest) and check which model gives you the best accuracy
5. Now use PCA to reduce dimensions, retrain your model and see what impact it has on your model in terms of accuracy. Keep in mind that many times doing PCA reduces the accuracy but computation is much lighter and that's the trade off you need to consider while building models in real life

In [247]:
import pandas as pd
df = pd.read_csv('heart.csv')
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [248]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [249]:
df = df[df.Cholesterol<(df.Cholesterol.mean()+3*df.Cholesterol.std())]
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [250]:
df = df[df.MaxHR<(df.MaxHR.mean()+3*df.MaxHR.std())]

In [251]:
df = df[df.FastingBS<=(df.FastingBS.mean()+3*df.FastingBS.std())]

In [252]:
df = df[df.Oldpeak<=(df.Oldpeak.mean()+3*df.Oldpeak.std())]

In [253]:
df.shape

(909, 12)

In [254]:
dir(df)

['Age',
 'ChestPainType',
 'Cholesterol',
 'ExerciseAngina',
 'FastingBS',
 'HeartDisease',
 'MaxHR',
 'Oldpeak',
 'RestingBP',
 'RestingECG',
 'ST_Slope',
 'Sex',
 'T',
 '_AXIS_LEN',
 '_AXIS_ORDERS',
 '_AXIS_TO_AXIS_NUMBER',
 '_HANDLED_TYPES',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__array_ufunc__',
 '__arrow_c_stream__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__dataframe__',
 '__deepcopy__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__divmod__',
 '__doc__',
 '__eq__',
 '__finalize__',
 '__firstlineno__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__iand__',
 '__ifloordiv__',
 '__imod__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__invert__',
 '__ior__',
 '__ipow__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__ixor__',
 '__le__',
 '__len__',
 '__lt__',
 '__matmul__',
 '__mod_

In [255]:
df.ChestPainType.unique()

<StringArray>
['ATA', 'NAP', 'ASY', 'TA']
Length: 4, dtype: str

In [256]:
df.RestingECG.unique()

<StringArray>
['Normal', 'ST', 'LVH']
Length: 3, dtype: str

In [257]:
df.ExerciseAngina.unique()

<StringArray>
['N', 'Y']
Length: 2, dtype: str

In [258]:
df.ST_Slope.unique()

<StringArray>
['Up', 'Flat', 'Down']
Length: 3, dtype: str

In [259]:
dfn = df.copy()

dfn['ST_Slope'] = dfn['ST_Slope'].replace({
    'Down': 1,
    'Flat': 2,
    'Up': 3
})

dfn['ExerciseAngina'] = dfn['ExerciseAngina'].replace({
    'N': 0,
    'Y': 1
})

dfn['RestingECG'] = dfn['RestingECG'].replace({
    'Normal': 1,
    'ST': 2,
    'LVH': 3
})
dfn.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,1,172,0,0.0,3,0
1,49,F,NAP,160,180,0,1,156,0,1.0,2,1
2,37,M,ATA,130,283,0,2,98,0,0.0,3,0
3,48,F,ASY,138,214,0,1,108,1,1.5,2,1
4,54,M,NAP,150,195,0,1,122,0,0.0,3,0


In [260]:
dfn = pd.get_dummies(dfn,drop_first=True,dtype=int)
dfn.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_2,RestingECG_3,ExerciseAngina_1,ST_Slope_2,ST_Slope_3
0,40,140,289,0,172,0.0,0,1,1,0,0,0,0,0,0,1
1,49,160,180,0,156,1.0,1,0,0,1,0,0,0,0,1,0
2,37,130,283,0,98,0.0,0,1,1,0,0,1,0,0,0,1
3,48,138,214,0,108,1.5,1,0,0,0,0,0,0,1,1,0
4,54,150,195,0,122,0.0,0,1,0,1,0,0,0,0,0,1


In [261]:
X = dfn.drop("HeartDisease",axis=1)
y = dfn.HeartDisease

In [262]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
Xs

array([[-1.43228867,  0.4140165 ,  0.85233413, ..., -0.82287409,
        -1.00110072,  1.14073039],
       [-0.47770182,  1.49623359, -0.16029364, ..., -0.82287409,
         0.99890049, -0.87663133],
       [-1.75048428, -0.12709205,  0.79659315, ..., -0.82287409,
        -1.00110072,  1.14073039],
       ...,
       [ 0.37081983, -0.12709205, -0.61551163, ...,  1.21525275,
         0.99890049, -0.87663133],
       [ 0.37081983, -0.12709205,  0.35995549, ..., -0.82287409,
         0.99890049, -0.87663133],
       [-1.64441908,  0.30579479, -0.20674445, ..., -0.82287409,
        -1.00110072,  1.14073039]], shape=(909, 15))

In [263]:
y = dfn.HeartDisease

In [264]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(Xs,y,test_size=0.2,random_state=10)

In [271]:
len(X_test)

182

In [272]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train,y_train)
model.score(X_test,y_test)

0.9010989010989011

In [275]:
from sklearn.decomposition import PCA
pca = PCA(0.95)
X_pca=pca.fit_transform(X)

In [277]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=30)
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier()
model_rf.fit(X_train_pca, y_train)
model_rf.score(X_test_pca, y_test)

0.7032967032967034